<a href="https://colab.research.google.com/github/appling2024/MSP/blob/daria-v/%D0%90%D1%84%D1%84%D0%B8%D0%BA%D1%81%D0%BD%D1%8B%D0%B9_%D1%82%D0%B5%D0%B3%D0%B3%D0%B5%D1%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [83]:
import nltk
nltk.download('punkt')
from nltk.tag import AffixTagger, UnigramTagger, BigramTagger
from nltk.corpus import treebank

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
!wget https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_train.txt
!wget https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_test.txt

--2024-11-07 20:56:31--  https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7626752 (7.3M) [text/plain]
Saving to: ‘GSD_train.txt’

GSD_train.txt       100%[===================>]   7.27M  --.-KB/s    in 0.1s    

2024-11-07 20:56:31 (72.7 MB/s) - ‘GSD_train.txt’ saved [7626752/7626752]

--2024-11-07 20:56:31--  https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_test.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 81386 (79K) [text

In [91]:
def load_data(filename):
    with open(filename, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    sentences = []
    current_sentence = []

    for line in lines:
        line = line.lower().strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) >= 4:
            word = parts[1]
            tag = parts[3]

            current_sentence.append((word, tag))

        else:
            print(f"Пропущена строка: {line} (недостаточно элементов)")

        if line.endswith('.'):
            current_sentence = [(word, tag) for word, tag in current_sentence if word.isalpha()]
            if current_sentence:
                sentences.append(current_sentence)
            current_sentence = []

    if current_sentence:
        current_sentence = [(word, tag) for word, tag in current_sentence if word.isalpha()]
        if current_sentence:
            sentences.append(current_sentence)

    return sentences

In [92]:
filename_train = 'GSD_train.txt'  # Обрабатываем обучающие данные (слово, тег)
train_data = load_data(filename_train)
filename_test = 'GSD_test.txt'  # Обрабатываем эталонную разметку (слово, тег)
etalon_data = load_data(filename_test)
# print(etalon_data)
# print(train_data)

In [93]:
# Создаем Affix Tagger и обучаем его
affix_tagger = AffixTagger(train_data)

In [109]:
# Обработка эталонной разметки
def extract_words_as_sentence(filename):
    words = []
    with open(filename, 'r', encoding='utf-8') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) > 1:
                word = parts[1]
                if word.isalpha():
                    words.append(word)

    sentence = ' '.join(words)
    return sentence

input_filename = 'GSD_test.txt'
result_sentence = extract_words_as_sentence(input_filename)

# print(result_sentence)
# print(len(result_sentence))

In [95]:
affix_tagger = AffixTagger(train_data)
tagged_words = affix_tagger.tag(nltk.word_tokenize(result_sentence))
print(tagged_words)

[('Билли', 'propn'), ('начал', 'verb'), ('играть', 'verb'), ('за', None), ('резервный', 'adj'), ('состав', 'noun'), ('Черка', 'noun'), ('в', None), ('возрасте', 'adv'), ('лет', None), ('а', None), ('через', 'adp'), ('пару', None), ('сезонов', 'noun'), ('был', None), ('приглашён', 'verb'), ('в', None), ('основной', 'adj'), ('состав', 'noun'), ('Стоимость', 'noun'), ('проезда', 'noun'), ('с', None), ('января', 'noun'), ('года', None), ('рублей', 'noun'), ('движение', 'noun'), ('осуществляется', 'verb'), ('с', None), ('до', None), ('Стал', None), ('членом', 'adj'), ('секретариата', 'noun'), ('общественной', 'adj'), ('безопасности', 'noun'), ('Мексики', 'noun'), ('SSP', None), ('Secretaría', None), ('de', None), ('Seguridad', None), ('Pública', 'x'), ('и', None), ('специальным', 'adj'), ('уполномоченным', 'adj'), ('Федеральной', 'adj'), ('полиции', 'noun'), ('Мексики', 'noun'), ('PFP', None), ('Policía', None), ('Federal', 'x'), ('Preventiva', None), ('Официальный', 'adj'), ('код', None), 

In [96]:
# Проверяем кол-во слов
print(len(tagged_words))

if isinstance(etalon_data, list) and len(etalon_data) == 1:
    nested_list = etalon_data[0]
else:
    nested_list = []
print(len(nested_list))


801
801


### Оцениваем точность теггера

In [97]:
import pandas as pd

def evaluate_tagging(tagged_words, nested_list):
    reference_dict = {word: tag for word, tag in nested_list}

    total_words = 0
    correct_predictions = 0
    mismatches = []
    comparison_data = []

    for word, tag in tagged_words:
        total_words += 1
        reference_tag = reference_dict.get(word)

        comparison_data.append({
            'Слово': word,
            'Тег в вашей разметке': tag,
            'Эталонный тег': reference_tag
        })

        if reference_tag == tag:
            correct_predictions += 1
        else:
            mismatches.append((word, tag, reference_tag))

    accuracy = correct_predictions / total_words if total_words > 0 else 0

    comparison_df = pd.DataFrame(comparison_data)

    mismatches_df = pd.DataFrame(mismatches, columns=['Слово', 'Тег в вашей разметке', 'Эталонный тег'])

    return accuracy, comparison_df, mismatches_df


accuracy, comparison_df, mismatches_df = evaluate_tagging(tagged_words, nested_list)

print(f"Точность: {accuracy:.2f}")
print("nТаблица сравнения (слово - тег в вашей разметке - эталонный тег):")
print(comparison_df)

if not mismatches_df.empty:
    print("nСлова с несовпадениями:")
    print(mismatches_df)
else:
    print("nНет слов с несовпадениями.")

Точность: 0.53
nТаблица сравнения (слово - тег в вашей разметке - эталонный тег):
         Слово Тег в вашей разметке Эталонный тег
0        Билли                propn          None
1        начал                 verb          verb
2       играть                 verb          verb
3           за                 None           adp
4    резервный                  adj           adj
..         ...                  ...           ...
796    несущие                 verb          verb
797        две                 None           num
798  хромосомы                 noun          noun
799          X                 None          None
800      самки                 noun          noun

[801 rows x 3 columns]
nСлова с несовпадениями:
         Слово Тег в вашей разметке Эталонный тег
0        Билли                propn          None
1           за                 None           adp
2        Черка                 noun          None
3            в                 None           adp
4     возрасте     

### Улучшаем точность теггера

In [110]:
class AffixTagger:
    def __init__(self, train_data):
        self.train_data = train_data
        self.affix_rules = self.create_affix_rules()

    def create_affix_rules(self):
        affix_rules = {}

        for sentence in self.train_data:
            for word, tag in sentence:
                # Извлекаем префиксы и суффиксы
                prefix = word[:3]
                suffix = word[-3:]

                # Правила для префиксов
                if prefix not in affix_rules:
                    affix_rules[prefix] = {}
                if tag not in affix_rules[prefix]:
                    affix_rules[prefix][tag] = 0
                affix_rules[prefix][tag] += 1

                # Правила для суффиксов
                if suffix not in affix_rules:
                    affix_rules[suffix] = {}
                if tag not in affix_rules[suffix]:
                    affix_rules[suffix][tag] = 0
                affix_rules[suffix][tag] += 1

        return affix_rules

    def tag(self, word):
        prefix = word[:3]
        suffix = word[-3:]

        if prefix in self.affix_rules:
            most_common_tag = max(self.affix_rules[prefix], key=self.affix_rules[prefix].get)
            return most_common_tag

        if suffix in self.affix_rules:
            most_common_tag = max(self.affix_rules[suffix], key=self.affix_rules[suffix].get)
            return most_common_tag

        return 'UNKNOWN'

filename_train = 'GSD_train.txt'
train_data = load_data(filename_train)

affix_tagger = AffixTagger(train_data)

input_filename = 'GSD_test.txt'
result_sentence = extract_words_as_sentence(input_filename)


tagged_words = [(word, affix_tagger.tag(word)) for word in nltk.word_tokenize(result_sentence)]
# print(tagged_words)

In [107]:
import pandas as pd

def evaluate_tagging(tagged_words, nested_list):
    reference_dict = {word: tag for word, tag in nested_list}

    total_words = 0
    correct_predictions = 0
    mismatches = []
    comparison_data = []

    for word, tag in tagged_words:
        total_words += 1
        reference_tag = reference_dict.get(word)

        comparison_data.append({
            'Слово': word,
            'Тег в вашей разметке': tag,
            'Эталонный тег': reference_tag
        })

        if reference_tag == tag:
            correct_predictions += 1
        else:
            mismatches.append((word, tag, reference_tag))

    accuracy = correct_predictions / total_words if total_words > 0 else 0

    comparison_df = pd.DataFrame(comparison_data)

    mismatches_df = pd.DataFrame(mismatches, columns=['Слово', 'Тег в вашей разметке', 'Эталонный тег'])

    return accuracy, comparison_df, mismatches_df


accuracy, comparison_df, mismatches_df = evaluate_tagging(tagged_words, nested_list)

print(f"Точность: {accuracy:.2f}")
print("nТаблица сравнения (слово - тег в вашей разметке - эталонный тег):")
print(comparison_df)

if not mismatches_df.empty:
    print("nСлова с несовпадениями:")
    print(mismatches_df)
else:
    print("nНет слов с несовпадениями.")

Точность: 0.56
nТаблица сравнения (слово - тег в вашей разметке - эталонный тег):
         Слово Тег в вашей разметке Эталонный тег
0        Билли                propn          None
1        начал                 verb          verb
2       играть                 noun          verb
3           за                  adp           adp
4    резервный                 noun           adj
..         ...                  ...           ...
796    несущие                  num          verb
797        две                  num           num
798  хромосомы                  adj          noun
799          X              UNKNOWN          None
800      самки                  adj          noun

[801 rows x 3 columns]
nСлова с несовпадениями:
         Слово Тег в вашей разметке Эталонный тег
0        Билли                propn          None
1       играть                 noun          verb
2    резервный                 noun           adj
3       состав                 verb          noun
4        Черка     